# Aula 01 - Lab 02: Agente com Ferramentas (Tools)

**Curso:** Agentic Engineering - SkillGo
**Prof.:** Ives Santos

No Lab 01 o modelo apenas **respondia**. Aqui ele passa a **agir**: consultar dados,
fazer calculos e executar operacoes atraves de *ferramentas* que nos definimos.

Roteiro:

1. Configuracao
2. O que e uma tool e como o LLM decide usa-la
3. Definindo ferramentas com `@tool`
4. `bind_tools`: o modelo pedindo uma ferramenta
5. O **loop do agente** na mao (para entender a mecanica)
6. Agente pronto com LangGraph (`create_react_agent`)
7. Memoria de conversa
8. Boas praticas, erros e aprovacao humana
9. Exercicios

> **Conceito-chave:** o LLM **nunca executa** a ferramenta. Ele apenas devolve um pedido
> estruturado (`tool_call`) com o nome da funcao e os argumentos. Quem executa e o **seu codigo**.
> Isso e o que torna o sistema auditavel e controlavel.

---
## 1. Configuracao

In [ ]:
# Descomente para instalar as dependencias
# %pip install -q langchain langchain-anthropic langgraph python-dotenv

In [ ]:
import os
import getpass

from dotenv import load_dotenv

load_dotenv()

if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("ANTHROPIC_API_KEY: ")

print("Chave configurada:", bool(os.environ.get("ANTHROPIC_API_KEY")))

In [ ]:
from langchain_anthropic import ChatAnthropic

MODEL_ID = "claude-opus-5"  # use "claude-sonnet-5" ou "claude-haiku-4-5" para reduzir custo

llm = ChatAnthropic(model=MODEL_ID, max_tokens=2048, timeout=60, max_retries=2)

print("LLM pronto:", MODEL_ID)

---
## 2. Por que ferramentas?

Um LLM sozinho tem limitacoes duras:

| Limitacao | Exemplo | Ferramenta resolve? |
|---|---|---|
| Nao sabe a data/hora atual | "Que dia e hoje?" | Sim |
| Erra em aritmetica longa | "17 * 4831 + 2209" | Sim |
| Nao acessa seus sistemas | "Qual o estoque do SKU 42?" | Sim |
| Nao executa acoes | "Abra um chamado" | Sim |
| Conhecimento com data de corte | "Qual a ultima release?" | Sim |

Vamos comprovar a segunda linha antes de dar ferramentas ao modelo.

In [ ]:
# Sem ferramenta: o modelo tenta calcular "de cabeca"
print(llm.invoke("Quanto e 4831 * 17 + 2209? Responda apenas o numero.").content)
print("Resposta correta:", 4831 * 17 + 2209)

---
## 3. Definindo ferramentas com `@tool`

O decorador `@tool` transforma uma funcao Python em uma ferramenta que o LLM enxerga.

**O que o modelo recebe (e o que importa muito):**
- o **nome** da funcao
- a **docstring** -> vira a descricao da ferramenta
- os **type hints** -> viram o schema JSON dos argumentos

> A docstring e o seu prompt para a ferramenta. Escreva **quando usar** e nao apenas
> o que ela faz. Descricoes vagas sao a causa numero 1 de agentes que erram a ferramenta.

In [ ]:
from datetime import datetime, timedelta

from langchain_core.tools import tool


@tool
def calculadora(expressao: str) -> str:
    """Avalia uma expressao aritmetica e retorna o resultado exato.

    Use SEMPRE que a pergunta envolver qualquer calculo numerico, por menor que pareca.
    Aceita apenas numeros e os operadores + - * / ( ) ** %.

    Args:
        expressao: expressao em sintaxe Python. Ex.: "4831 * 17 + 2209"
    """
    permitidos = set("0123456789+-*/(). %")
    if not set(expressao) <= permitidos:
        return "Erro: a expressao contem caracteres nao permitidos."
    try:
        return str(eval(expressao, {"__builtins__": {}}, {}))
    except Exception as e:
        return f"Erro ao calcular: {e}"


@tool
def data_atual(formato: str = "%d/%m/%Y") -> str:
    """Retorna a data e hora atuais do sistema.

    Use quando a pergunta depender de "hoje", "agora", "esta semana" ou qualquer
    referencia temporal relativa.

    Args:
        formato: string de formatacao strftime. Padrao: dia/mes/ano.
    """
    return datetime.now().strftime(formato)


print(calculadora.name)
print(calculadora.description)
print(calculadora.args_schema.model_json_schema())

### 3.1 Ferramentas que simulam sistemas internos

Em producao estas funcoes chamariam um banco, uma API REST ou um servico interno.
Aqui usamos dicionarios em memoria para focar no comportamento do agente.

In [ ]:
ESTOQUE = {
    "SKU-100": {"nome": "Teclado mecanico", "quantidade": 42, "preco": 349.90},
    "SKU-200": {"nome": "Monitor 27 polegadas", "quantidade": 7, "preco": 1899.00},
    "SKU-300": {"nome": "Headset com cancelamento", "quantidade": 0, "preco": 799.50},
}

CHAMADOS = []


@tool
def consultar_estoque(sku: str) -> str:
    """Consulta nome, quantidade disponivel e preco de um produto pelo codigo SKU.

    Use quando o usuario perguntar sobre disponibilidade, preco ou detalhes de um produto.

    Args:
        sku: codigo do produto no formato "SKU-000". Ex.: "SKU-100"
    """
    item = ESTOQUE.get(sku.upper().strip())
    if not item:
        return f"SKU '{sku}' nao encontrado. SKUs validos: {', '.join(ESTOQUE)}"
    return (
        f"{sku.upper()} | {item['nome']} | "
        f"quantidade: {item['quantidade']} | preco unitario: R$ {item['preco']:.2f}"
    )


@tool
def listar_produtos() -> str:
    """Lista todos os SKUs cadastrados com seus nomes.

    Use quando o usuario nao souber o codigo do produto ou pedir o catalogo completo.
    """
    return "\n".join(f"{sku}: {d['nome']}" for sku, d in ESTOQUE.items())


@tool
def abrir_chamado(titulo: str, descricao: str, prioridade: str = "media") -> str:
    """Abre um chamado de suporte interno e retorna o numero do protocolo.

    Use apenas quando o usuario pedir EXPLICITAMENTE para abrir, registrar ou criar
    um chamado/ticket. Nunca abra um chamado por conta propria.

    Args:
        titulo: resumo curto do problema.
        descricao: detalhamento do problema.
        prioridade: "baixa", "media" ou "alta".
    """
    if prioridade not in {"baixa", "media", "alta"}:
        return "Erro: prioridade deve ser 'baixa', 'media' ou 'alta'."
    protocolo = f"CH-{1000 + len(CHAMADOS)}"
    CHAMADOS.append(
        {
            "protocolo": protocolo,
            "titulo": titulo,
            "descricao": descricao,
            "prioridade": prioridade,
            "aberto_em": datetime.now().isoformat(timespec="seconds"),
        }
    )
    return f"Chamado {protocolo} aberto com prioridade {prioridade}."


ferramentas = [calculadora, data_atual, consultar_estoque, listar_produtos, abrir_chamado]

print(f"{len(ferramentas)} ferramentas registradas:")
for f in ferramentas:
    print(" -", f.name)

---
## 4. `bind_tools`: o modelo pedindo uma ferramenta

`bind_tools` anexa os schemas das ferramentas a cada chamada do modelo.
Repare no que acontece: o `content` vem vazio (ou com um comentario) e o campo
`tool_calls` traz o **pedido estruturado**.

In [ ]:
llm_com_tools = llm.bind_tools(ferramentas)

resposta = llm_com_tools.invoke("Quanto e 4831 * 17 + 2209?")

print("content   :", repr(resposta.content))
print("tool_calls:", resposta.tool_calls)

### 4.1 Executando o pedido manualmente

O `tool_call` e um dicionario com `name`, `args` e `id`. Nos localizamos a ferramenta
pelo nome e a invocamos. O resultado volta ao modelo como uma `ToolMessage`,
**amarrada pelo `tool_call_id`**.

In [ ]:
from langchain_core.messages import HumanMessage, ToolMessage

mapa_ferramentas = {f.name: f for f in ferramentas}

mensagens = [HumanMessage("Quanto e 4831 * 17 + 2209?")]
ai_msg = llm_com_tools.invoke(mensagens)
mensagens.append(ai_msg)

for chamada in ai_msg.tool_calls:
    ferramenta = mapa_ferramentas[chamada["name"]]
    resultado = ferramenta.invoke(chamada["args"])
    print(f"-> executando {chamada['name']}({chamada['args']}) = {resultado}")
    mensagens.append(ToolMessage(content=str(resultado), tool_call_id=chamada["id"]))

final = llm_com_tools.invoke(mensagens)
print("\nResposta final:", final.content)

### 4.2 Chamadas paralelas

Quando as ferramentas sao independentes, o modelo pode pedir varias **na mesma resposta**.
Isso reduz o numero de idas e voltas ao LLM.

In [ ]:
r = llm_com_tools.invoke(
    "Quantas unidades temos do SKU-100 e do SKU-200? E que dia e hoje?"
)

for c in r.tool_calls:
    print(f"{c['name']}({c['args']})")

---
## 5. O loop do agente, na mao

Um agente e, essencialmente, este laco:

```
1. Enviar mensagens + ferramentas ao LLM
2. Se a resposta tiver tool_calls:
       executar cada ferramenta
       anexar os resultados como ToolMessage
       voltar ao passo 1
3. Senao: a resposta final chegou. Encerrar.
```

O `max_iteracoes` e o seu **freio de emergencia** contra loops infinitos.

In [ ]:
from langchain_core.messages import SystemMessage


def agente_simples(pergunta: str, max_iteracoes: int = 6, verbose: bool = True) -> str:
    """Implementacao didatica do loop de um agente ReAct."""
    mensagens = [
        SystemMessage(
            "Voce e o assistente operacional da SkillGo. "
            "Use as ferramentas disponiveis sempre que precisar de dados reais - "
            "nunca invente numeros, precos ou datas. Responda em portugues, de forma objetiva."
        ),
        HumanMessage(pergunta),
    ]

    for i in range(1, max_iteracoes + 1):
        ai_msg = llm_com_tools.invoke(mensagens)
        mensagens.append(ai_msg)

        if not ai_msg.tool_calls:
            if verbose:
                print(f"[iteracao {i}] resposta final\n")
            return ai_msg.content

        for chamada in ai_msg.tool_calls:
            ferramenta = mapa_ferramentas.get(chamada["name"])
            if ferramenta is None:
                resultado = f"Erro: ferramenta '{chamada['name']}' nao existe."
            else:
                try:
                    resultado = ferramenta.invoke(chamada["args"])
                except Exception as e:
                    # Devolver o erro ao modelo permite que ele se corrija
                    resultado = f"Erro ao executar a ferramenta: {e}"

            if verbose:
                print(f"[iteracao {i}] {chamada['name']}({chamada['args']}) -> {resultado}")

            mensagens.append(
                ToolMessage(content=str(resultado), tool_call_id=chamada["id"])
            )

    return "Limite de iteracoes atingido sem resposta final."


print(agente_simples("Qual o valor total se eu comprar 3 unidades do SKU-200?"))

In [ ]:
print(agente_simples("O headset esta disponivel? Se nao estiver, abra um chamado de alta prioridade pedindo reposicao."))

In [ ]:
# Conferindo o efeito colateral real da acao do agente
CHAMADOS

---
## 6. Agente pronto com LangGraph

Na pratica voce nao escreve o loop na mao. O `create_react_agent` do **LangGraph** entrega
o mesmo comportamento com tratamento de erros, streaming e estado.

O nome **ReAct** vem de *Reasoning + Acting*: o modelo alterna entre raciocinar sobre o
que falta e agir chamando ferramentas.

In [ ]:
from langgraph.prebuilt import create_react_agent

PROMPT_SISTEMA = (
    "Voce e o assistente operacional da SkillGo.\n"
    "- Use as ferramentas para obter dados reais; nunca invente numeros, precos ou datas.\n"
    "- Faca todos os calculos com a ferramenta calculadora.\n"
    "- So abra um chamado se o usuario pedir explicitamente.\n"
    "- Responda em portugues, de forma objetiva."
)

agente = create_react_agent(
    model=llm,
    tools=ferramentas,
    prompt=PROMPT_SISTEMA,
)

resultado = agente.invoke(
    {"messages": [HumanMessage("Quanto custam 5 teclados mecanicos? Liste os produtos se precisar.")]}
)

print(resultado["messages"][-1].content)

### 6.1 Inspecionando o raciocinio

`resultado["messages"]` guarda a conversa inteira: pedidos de ferramenta, resultados e
resposta final. Esse historico e a sua principal ferramenta de **debug de agentes**.

In [ ]:
for m in resultado["messages"]:
    tipo = type(m).__name__
    if getattr(m, "tool_calls", None):
        for c in m.tool_calls:
            print(f"{tipo:<12} CHAMA {c['name']}({c['args']})")
    elif tipo == "ToolMessage":
        print(f"{tipo:<12} RETORNO -> {m.content}")
    elif m.content:
        print(f"{tipo:<12} {str(m.content)[:120]}")

### 6.2 Streaming dos passos

Em uma UI real voce mostra o progresso do agente enquanto ele trabalha,
em vez de deixar o usuario olhando para uma tela parada.

In [ ]:
entrada = {"messages": [HumanMessage("Temos monitores em estoque? Quanto sairiam 4 unidades?")]}

for passo in agente.stream(entrada, stream_mode="values"):
    ultima = passo["messages"][-1]
    if getattr(ultima, "tool_calls", None):
        for c in ultima.tool_calls:
            print(f"[acao]      {c['name']}({c['args']})")
    elif type(ultima).__name__ == "ToolMessage":
        print(f"[observacao] {ultima.content}")
    elif type(ultima).__name__ == "AIMessage" and ultima.content:
        print(f"[resposta]   {ultima.content}")

---
## 7. Memoria de conversa

Por padrao cada `invoke` e independente - o agente nao lembra do turno anterior.
O `checkpointer` do LangGraph persiste o estado por `thread_id`, permitindo
conversas com continuidade.

In [ ]:
from langgraph.checkpoint.memory import MemorySaver

agente_com_memoria = create_react_agent(
    model=llm,
    tools=ferramentas,
    prompt=PROMPT_SISTEMA,
    checkpointer=MemorySaver(),
)

config = {"configurable": {"thread_id": "conversa-turma-01"}}


def conversar(texto: str) -> None:
    saida = agente_com_memoria.invoke({"messages": [HumanMessage(texto)]}, config=config)
    print(f"Usuario: {texto}")
    print(f"Agente : {saida['messages'][-1].content}\n")


conversar("Quantos monitores temos em estoque?")
conversar("E qual o preco unitario deles?")   # 'deles' so faz sentido com memoria
conversar("Entao quanto sairiam 3 unidades?")

---
## 8. Boas praticas

### 8.1 Descricao da ferramenta e prompt

Compare as duas descricoes abaixo. A primeira produz agentes erraticos:

```python
@tool
def buscar(q: str) -> str:
    """Busca coisas."""          # ruim: quando usar? o que e 'q'?
```

```python
@tool
def buscar_pedido(numero_pedido: str) -> str:
    """Retorna status, data e itens de um pedido pelo numero.

    Use quando o usuario mencionar um numero de pedido ou perguntar
    "onde esta meu pedido". Nao use para consultas de estoque.

    Args:
        numero_pedido: numero no formato "PED-0000".
    """
```

**Regra pratica:** descreva *quando usar* e *quando NAO usar*.

### 8.2 Erros sao informacao, nao excecao

Uma ferramenta que levanta excecao derruba o agente. Uma ferramenta que **retorna a
mensagem de erro** da ao modelo a chance de corrigir o proprio caminho.

In [ ]:
# O SKU nao existe: a ferramenta retorna um erro util e o agente se recupera sozinho
print(agente.invoke({"messages": [HumanMessage("Quantas unidades do SKU-999 temos?")]})["messages"][-1].content)

### 8.3 Aprovacao humana (human-in-the-loop)

Acoes com efeito colateral - abrir chamado, enviar e-mail, apagar dados, mover dinheiro -
nao devem executar sozinhas. O padrao e **interceptar** a chamada, pedir confirmacao e
so entao executar.

In [ ]:
FERRAMENTAS_SENSIVEIS = {"abrir_chamado"}


def executar_com_aprovacao(chamada: dict, aprovar_automaticamente: bool = False) -> str:
    """Executa uma tool_call, exigindo confirmacao para ferramentas sensiveis."""
    nome, args = chamada["name"], chamada["args"]

    if nome in FERRAMENTAS_SENSIVEIS and not aprovar_automaticamente:
        print(f"[APROVACAO NECESSARIA] {nome}({args})")
        resposta = input("Autorizar? (s/n): ").strip().lower()
        if resposta != "s":
            # Devolver a recusa ao modelo, e nao um erro tecnico
            return "Acao recusada pelo usuario. Nao tente novamente sem nova autorizacao."

    return str(mapa_ferramentas[nome].invoke(args))


# Simulacao (sem input interativo)
print(executar_com_aprovacao(
    {"name": "consultar_estoque", "args": {"sku": "SKU-100"}}
))

### 8.4 Checklist para producao

- [ ] Cada ferramenta tem docstring dizendo **quando usar** e **quando nao usar**
- [ ] Todos os argumentos sao validados dentro da funcao (nao confie no LLM)
- [ ] Erros retornam texto util em vez de levantar excecao
- [ ] Existe limite de iteracoes e timeout por ferramenta
- [ ] Acoes destrutivas ou irreversiveis passam por aprovacao humana
- [ ] Toda chamada de ferramenta e registrada em log (auditoria)
- [ ] Poucas ferramentas e bem delimitadas; evite dezenas com escopos sobrepostos
- [ ] Nunca exponha credenciais ou dados sensiveis como argumento de ferramenta

---
## 9. Exercicios

1. **Nova ferramenta.** Crie `atualizar_estoque(sku, quantidade)` que altera o dicionario
   `ESTOQUE`. Marque-a como sensivel e teste o fluxo de aprovacao.
2. **Ferramenta com data.** Crie `dias_ate(data_alvo)` que calcula quantos dias faltam para
   uma data. Peca ao agente: "quantos dias faltam para o Natal?".
3. **Descricao ruim.** Troque a docstring de `calculadora` por apenas "Calcula." e refaca
   as perguntas. Documente quantas vezes o agente deixou de usar a ferramenta.
4. **Limite.** Force um loop: crie uma ferramenta que sempre responde "tente novamente".
   Observe o `max_iteracoes` agindo como freio.
5. **RAG + Tools.** Transforme o `retriever` do Lab 01 em uma ferramenta
   (`buscar_politica_interna`) e adicione-a a lista de `ferramentas`. Voce tera um agente
   que consulta documentos **e** sistemas.
6. **Observabilidade.** Adicione um `print` (ou `logging`) em cada ferramenta registrando
   argumentos e tempo de execucao.

---
## Resumo

| Conceito | Componente |
|---|---|
| Definir ferramenta | decorador `@tool` |
| Expor ferramentas ao LLM | `llm.bind_tools(ferramentas)` |
| Pedido do modelo | `AIMessage.tool_calls` |
| Devolver resultado | `ToolMessage(content=..., tool_call_id=...)` |
| Agente pronto | `create_react_agent` (LangGraph) |
| Memoria | `MemorySaver` + `thread_id` |
| Passo a passo | `agente.stream(..., stream_mode="values")` |

**O que muda de mentalidade:** o LLM deixa de ser um gerador de texto e passa a ser um
**orquestrador** de funcoes que voce escreveu, testou e controla. A qualidade do agente
depende muito mais das suas ferramentas e descricoes do que do modelo escolhido.